# Week 9 · Day 3 — Is the scorecard any good?

HW 9.3, the validation pack: six functions (TODO + an assert cell under each), the tables by product and by year, and five sentences for Nilufar. The assert cells check your result without showing it. Finish with **Kernel → Restart & Run All**.

In [1]:
from pathlib import Path
import hashlib
import sqlite3
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")           # this notebook lives in data/python/week9/, so "../.." is the data/ folder
con = sqlite3.connect(DATA / "raw" / "nur.db")


def check(value, fingerprints, dp=0):
    """Right or wrong without showing the answer: the rounded value's fingerprint must be one of the expected ones."""
    fp = hashlib.sha256(f"nur-w9:{round(float(value), dp):.{dp}f}".encode()).hexdigest()[:12]
    assert fp in fingerprints, "not yet: this result is not the expected one — check your function and run again"

In [2]:
# The scored loans of the week (as on the Day 3 page; run, do not edit)
L = pd.read_sql("""
SELECT l.loan_id, l.disbursed_at, l.risk_grade, l.status, p.code AS product, a.credit_score AS score,
       CASE WHEN l.status IN ('default', 'written_off') THEN 1 ELSE 0 END AS bad
FROM loans l
JOIN applications a ON a.application_id = l.application_id
JOIN products p ON p.product_id = l.product_id
ORDER BY l.loan_id""", con)
S = L.dropna(subset=["score"]).copy()
S["year"] = S["disbursed_at"].str[:4]
s, y = S["score"].to_numpy(), S["bad"].to_numpy()
assert (len(L), L.bad.sum(), len(S), y.sum()) == (7828, 234, 7615, 223)
print(len(S), "scored loans,", y.sum(), "bads")

7615 scored loans, 223 bads


## 1. `confusion_at(score, bad, cut)` — the four counts for "decline below cut"
Return `(TP, FP, FN, TN)`: bads declined, goods declined, bads approved, goods approved.

In [3]:
def confusion_at(score, bad, cut):
    """(TP, FP, FN, TN) when every score below cut is declined and the bad is the positive class."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [4]:
tp, fp, fn, tn = confusion_at(s, y, 640)
check(tp, ['2d5830ce13ff', '77e555afe289', 'd80754e1eed1'], 0); check(fp, ['88aec61d0634', 'ba9c5b00e41e', 'ecbcde16a8d2'], 0); check(fn, ['94bb9802966b', 'd655b0bd598f', 'fd2cf662de0a'], 0); check(tn, ['551f5e7f45df', '8c2468faacbb', 'cf44b2d8ed10'], 0)
print("confusion_at OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 2. `roc_points(score, bad)` — FPR and TPR for every cut-off
One cut-off per score from the lowest to one above the highest; return two numpy arrays `(fpr, tpr)`, starting at (0, 0) and ending at (1, 1).

In [5]:
def roc_points(score, bad):
    """Arrays (fpr, tpr) over every cut-off, from approve-everyone to decline-everyone."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [6]:
fpr, tpr = roc_points(s, y)
assert fpr[0] == 0 and tpr[0] == 0 and fpr[-1] == 1 and tpr[-1] == 1, "the curve must run from (0, 0) to (1, 1)"
area = np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)
check(area, ['1d41ca959cbc', 'e9d2a91f4f7f', 'f7a6cbd46b88'], 3)
print("roc_points OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 3. `auc_by_ranks(score, bad)` — the AUC from ranks, a tie counted half
Use `stats.rankdata`; a high score means a good loan.

In [7]:
def auc_by_ranks(score, bad):
    """P(a random good scores above a random bad), ties counted half — the Mann-Whitney U over the number of pairs."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [8]:
assert auc_by_ranks(np.array([1, 2, 3, 4]), np.array([1, 1, 0, 0])) == 1.0
assert auc_by_ranks(np.array([5, 5]), np.array([1, 0])) == 0.5, "a tie counts half"
check(auc_by_ranks(s, y), ['529b5f01b8d2', '90d718da9c16', 'ad126cc33281'], 4)
print("auc_by_ranks OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 4. `ks_stat(score, bad)` — the widest gap and where it is
Return `(ks, x)`: the largest (share of bads scoring ≤ x) − (share of goods scoring ≤ x), and the score x where it is reached.

In [9]:
def ks_stat(score, bad):
    """(KS, score) — the widest gap between the cumulative shares of bads and goods, and where it is reached."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [10]:
ks, at = ks_stat(s, y)
check(ks, ['2ed4aae4491e', '6181c0e64e7f', '71cfffeb4ab9'], 4); check(at, ['3404530db3d7', '3a7f2d4c0892', 'c09f3230adb8'], 0)
print("ks_stat OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 5. `gains_table(frame, groups=10)` — loans, bads, gain and lift per decile
Sort by `score` and then `loan_id` (so ties split the same way every time), deal the loans into `groups` equal groups, worst first, as SQL's `NTILE` does (when they do not divide evenly, the first groups take one loan more). Columns: `loans`, `bads`, `bad_rate`, `gain` (cumulative share of all bads, 0–1) and `lift` (the group's bad rate ÷ the overall bad rate).

In [11]:
def gains_table(frame, groups=10):
    """One row per group of the score, worst first: loans, bads, bad_rate, gain (cumulative share of bads), lift."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [12]:
g = gains_table(S)
assert len(g) == 10 and g["loans"].sum() == 7615
check(g["bads"].iloc[0], ['181b07afa2e5', '1cd49cb8ca7f', '9d0b73d023da'], 0); check(g["gain"].iloc[2], ['47d8915ae006', '694c13cbd4f3', 'a86ee55543c9'], 3); check(g["lift"].iloc[0], ['68efb50754e4', '73aef3387893', '7e0674dc43a8'], 2)
print("gains_table OK")
g.round(3)

NotImplementedError: TODO: write me, then run the assert cell below

## 6. `validation_table(frame, by)` — the whole check, per segment
One row per value of the column `by`: `loans`, `bads`, `auc`, `gini`, `ks` (use your functions above).

In [13]:
def validation_table(frame, by):
    """loans, bads, AUC, Gini and KS of the score for each value of the column `by`."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")


In [14]:
vp = validation_table(S, "product")
vy = validation_table(S, "year")
assert len(vp) == 5 and len(vy) == 3
check(vp.loc["AUTO", "gini"], ['237ab4d69869', '991634018d2a', 'bb6afd1e265b'], 3); check(vp.loc["SME", "ks"], ['60c192ee9741', '78ae3d0728da', 'd6b4c414506d'], 3); check(vy.loc["2026", "bads"], ['5b0f9035fb83', '681fd83ac428', 'a449a9b77ec8'], 0); check(vy.loc["2024", "gini"], ['013eab721ff9', '6254c0d9232a', 'fe8e374ddbce'], 3)
print("validation_table OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 7. The two tables for Nilufar
Run it: by product, then by year of disbursal. These are the numbers for the five sentences (and for H1–H3 on the page).

In [15]:
print(validation_table(S, "product").round(3))
print()
print(validation_table(S, "year").round(3))

NotImplementedError: TODO: write me, then run the assert cell below

## 8. The ROC curve, once more (optional)
Our score against a coin toss, with the KS point marked.

In [16]:
fpr, tpr = roc_points(s, y)
j = np.argmax(tpr - fpr)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label="credit score")
plt.plot([0, 1], [0, 1], "--", label="coin toss")
plt.plot([fpr[j], fpr[j]], [fpr[j], tpr[j]], lw=3, label="KS gap")
plt.xlabel("FPR: goods turned away"); plt.ylabel("TPR: bads caught"); plt.legend(); plt.show()

NotImplementedError: TODO: write me, then run the assert cell below

## 9. Five sentences for Nilufar
The Gini (or AUC) and the KS with its score · the weakest product with its number · the out-of-time picture, and why 2026 is uncertain · one recommendation.

*(write here)*